[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/098.%20The%20Green%20Vehicle%20Routing%20Problem/P98-Tier-8_executed.ipynb)

---


# 98. The Green Vehicle Routing Problem

## Tier 8: Algorithmic Fairness and Constitutional Constraints

### Purpose and scope

This notebook demonstrates how a small set of candidate vehicle routes can be assessed against environmental-equity principles and stakeholder preferences. The procedure generates three specified candidate plans, evaluates each one, and selects the highest-scoring compliant plan. If none is compliant, it selects the plan with the smallest total penalty, using the combined score to break ties. This is a transparent demonstration procedure, not a general-purpose route optimizer: it does not search every possible route or prove global optimality.

The inputs are a distance matrix, an arc-emissions matrix, neighborhood and stakeholder descriptions, a fleet of four vehicles, and a baseline set of routes. The outputs are the selected candidate, its cost and emissions, modeled neighborhood emissions burdens, compliance assessments, stakeholder utilities, and diagnostic plots.

### Modeling assumptions

- Nodes are indexed from 0 to 8. Node 0 is the depot; nodes 1–8 correspond to the eight neighborhoods. Every candidate visits each neighborhood exactly once and returns each route to the depot.
- Distances are in kilometers. The supplied emissions matrix gives emissions in kilograms of CO₂ for each traversed arc. These arc emissions are fixed inputs: vehicle type does not change them in this example.
- Diesel/electric vehicle types are assigned to routes in list order. They affect the vehicle-equity metric, but not the supplied arc-emissions values.
- Neighborhood burden is a sensitivity-weighted allocation of route-segment CO₂ emissions. It is an illustrative route-related burden proxy, not a prediction of local PM2.5 concentration. The recorded baseline PM2.5 values are contextual data and are not added to or compared numerically with CO₂ emissions.
- Neighborhoods with income below the median neighborhood income are classified as disadvantaged for the vehicle-equity calculation. The demographic and income data are descriptive inputs; they do not establish causal effects or represent a complete environmental-justice assessment.
- Service quality is the fraction of the eight neighborhood nodes visited. No time windows, vehicle capacities, congestion, charging constraints, or uncertainty are modeled.

### Concepts and mathematical model

A **candidate plan** is a set of routes generated by the notebook's prescribed baseline, equity-focused, and hybrid constructions. A **constitutional principle** is a threshold rule. A lower-is-better measure complies when it does not exceed its threshold; a higher-is-better measure complies when it reaches or exceeds its threshold. A penalty records the size of a threshold violation multiplied by the principle's penalty rate.

Let $N=\{1,\ldots,8\}$ be the neighborhood nodes, let $R$ be the set of routes, and let $r_k=(v_{k,0},\ldots,v_{k,m_k})$ denote route $k$. For arc $(i,j)$, let $d_{ij}$ be distance and $e_{ij}$ be the supplied CO₂ emissions. For neighborhood $n$, let $s_n$ be its sensitivity factor. The modeled burden is

$$B_n=\sum_{k\in R}\sum_{t=0}^{m_k-1}\frac{e_{v_{k,t},v_{k,t+1}}}{|A_{k,t}|}\,s_n\,\mathbf{1}[n\in A_{k,t}],$$

where $A_{k,t}$ is the set of neighborhoods considered affected by the segment's endpoints, and $\mathbf{1}[\cdot]$ is 1 when its condition holds and 0 otherwise. The implementation allocates each segment's emissions equally across its affected neighborhoods before applying sensitivity. The equitable-exposure measure is $\max_n B_n / \overline{B}$, where $\overline{B}$ is the mean burden; it complies when this ratio is at most 1.2.

For sorted burdens $B_{(1)}\leq\cdots\leq B_{(q)}$, the Gini coefficient and the notebook's fairness score are

$$G=\frac{2\sum_{i=1}^{q}iB_{(i)}}{q\sum_{i=1}^{q}B_{(i)}}-\frac{q+1}{q},\qquad F=1-G.$$

Here, $q$ is the number of neighborhoods. The score $F$ lies between 0 and 1 for nonnegative burdens: a larger value indicates a more even distribution. If total burden is zero, the implementation defines $G=0$ and $F=1$.

Let $C$ denote plan cost, $E$ its total emissions, and $C_0,E_0$ the cost and emissions of the baseline routes at the baseline cost rate. The candidate score is

$$S=\sum_{h\in H}U_h+100F-\sum_{p\in P}\pi_p,$$

where $H$ is the stakeholder set, $U_h$ is stakeholder $h$'s weighted, dimensionless utility, $P$ is the set of principles, and $\pi_p$ is the principle's penalty in illustrative policy points. Stakeholder cost and emissions terms are normalized by $C_0$ and $E_0$; equity and service terms use their respective scores. The factor 100 and penalty rates are policy weights for this demonstration, not monetary values. Among compliant candidates, the procedure selects the largest $S$. When no candidate complies, it first minimizes total penalty and then uses $S$ as a tie-breaker.

The three-candidate evaluation is finite and small: if there are $K$ candidate plans, $q$ neighborhoods, and $L$ total route segments across a plan, burden calculation takes $O(Lq)$ time with the simple neighborhood lookup, and sorting for the Gini score takes $O(q\log q)$. The candidate-generation procedure does not enumerate all possible vehicle-routing solutions.

### What to examine

Compare route-related burden across neighborhoods, identify which thresholds are met, and inspect how candidate cost, emissions, equity, and stakeholder utility differ. A low penalty is not the same as full compliance, and a favorable score is meaningful only relative to the stated assumptions and policy weights.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")
np.random.seed(42)


In [ ]:
class ConstitutionalPrinciple:
    """A threshold rule and its penalty for noncompliance."""

    def __init__(
        self,
        name: str,
        description: str,
        weight: float,
        threshold: float,
        penalty: float,
        higher_is_better: bool = False,
    ) -> None:
        self.name = name
        self.description = description
        self.weight = weight
        self.threshold = threshold
        self.penalty = penalty
        self.higher_is_better = higher_is_better

    def evaluate_compliance(self, value: float) -> Tuple[bool, float]:
        """Return compliance and threshold-violation penalty points."""
        if self.higher_is_better:
            violation_amount = max(0.0, self.threshold - value)
        else:
            violation_amount = max(0.0, value - self.threshold)
        return violation_amount == 0.0, violation_amount * self.penalty


@dataclass
class Neighborhood:
    """Neighborhood attributes used by the illustrative assessment."""
    id: int
    name: str
    population: int
    median_income: float
    demographic_composition: Dict[str, float]
    baseline_exposure: float
    geographic_coords: Tuple[float, float]
    sensitivity_factor: float


@dataclass
class Stakeholder:
    """Stakeholder preferences and descriptive influence level."""
    name: str
    type: str
    utility_weights: Dict[str, float]
    primary_concerns: List[str]
    influence_level: float


@dataclass
class RoutePlan:
    """Candidate routes and their calculated evaluation measures."""
    id: int
    name: str
    routes: List[List[int]]
    total_distance: float
    total_emissions: float
    total_cost: float
    neighborhood_exposures: Dict[int, float]
    equity_metrics: Dict[str, float]
    constitutional_compliance: Dict[str, Tuple[bool, float]]
    stakeholder_utilities: Dict[str, float]
    fairness_score: float


class ConstitutionalAI:
    """Evaluate prescribed route candidates using principles and utilities."""

    def __init__(
        self, neighborhoods: List[Neighborhood], stakeholders: List[Stakeholder]
    ) -> None:
        self.neighborhoods = {n.id: n for n in neighborhoods}
        self.stakeholders = {s.name: s for s in stakeholders}
        self.baseline_cost = None
        self.baseline_emissions = None

        self.principles = {
            "equitable_exposure": ConstitutionalPrinciple(
                "Equitable Exposure",
                "No neighborhood should exceed 120% of mean modeled route-related burden.",
                0.3, 1.2, 1000.0,
            ),
            "vehicle_equity": ConstitutionalPrinciple(
                "Vehicle Equity",
                "The electric-mile share in disadvantaged areas should meet the fleet share.",
                0.2, 1.0, 500.0, higher_is_better=True,
            ),
            "economic_viability": ConstitutionalPrinciple(
                "Economic Viability",
                "Total cost should not exceed 150% of baseline routing cost.",
                0.2, 1.5, 200.0,
            ),
            "service_quality": ConstitutionalPrinciple(
                "Service Quality",
                "At least 90% of neighborhood nodes should be served.",
                0.3, 0.9, 300.0, higher_is_better=True,
            ),
        }

    def _get_neighborhoods_on_route(
        self, from_node: int, to_node: int
    ) -> List[int]:
        """Map an arc to neighborhoods adjacent to either endpoint."""
        neighborhood_nodes = {
            1: [1, 2], 2: [2, 3], 3: [3, 4], 4: [4, 5],
            5: [5, 6], 6: [6, 7], 7: [7, 8], 8: [8, 1],
        }
        return [
            n_id for n_id, nodes in neighborhood_nodes.items()
            if from_node in nodes or to_node in nodes
        ]

    def calculate_neighborhood_exposures(
        self,
        routes: List[List[int]],
        emissions: np.ndarray,
    ) -> Dict[int, float]:
        """Allocate route-segment emissions as sensitivity-weighted burdens."""
        burdens = {n_id: 0.0 for n_id in self.neighborhoods}
        for route in routes:
            for from_node, to_node in zip(route[:-1], route[1:]):
                affected = self._get_neighborhoods_on_route(from_node, to_node)
                if not affected:
                    continue
                segment_emissions = float(emissions[from_node, to_node])
                share = segment_emissions / len(affected)
                for n_id in affected:
                    burdens[n_id] += share * self.neighborhoods[n_id].sensitivity_factor
        return burdens

    def _calculate_fairness_score(self, route_plan: RoutePlan) -> float:
        """Return one minus the Gini coefficient of neighborhood burdens."""
        values = np.sort(np.asarray(list(route_plan.neighborhood_exposures.values())))
        count = len(values)
        total = float(values.sum())
        if count == 0:
            return 1.0
        if total <= 0.0:
            return 1.0
        ranks = np.arange(1, count + 1, dtype=float)
        gini = (2.0 * np.sum(ranks * values) / (count * total)) - (count + 1.0) / count
        return float(np.clip(1.0 - gini, 0.0, 1.0))

    def _vehicle_equity_ratio(
        self, routes: List[List[int]], distances: np.ndarray, vehicle_types: List[str]
    ) -> float:
        """Compare electric-mile share in disadvantaged areas with fleet share."""
        incomes = [n.median_income for n in self.neighborhoods.values()]
        income_cutoff = float(np.median(incomes))
        disadvantaged = {
            n_id for n_id, neighborhood in self.neighborhoods.items()
            if neighborhood.median_income < income_cutoff
        }
        total_miles = 0.0
        electric_miles = 0.0
        disadvantaged_miles = 0.0
        disadvantaged_electric_miles = 0.0

        for route, vehicle_type in zip(routes, vehicle_types):
            is_electric = vehicle_type == "electric"
            for from_node, to_node in zip(route[:-1], route[1:]):
                distance = float(distances[from_node, to_node])
                total_miles += distance
                if is_electric:
                    electric_miles += distance
                affected = set(self._get_neighborhoods_on_route(from_node, to_node))
                if affected.intersection(disadvantaged):
                    disadvantaged_miles += distance
                    if is_electric:
                        disadvantaged_electric_miles += distance

        fleet_share = electric_miles / total_miles if total_miles else 0.0
        area_share = (
            disadvantaged_electric_miles / disadvantaged_miles
            if disadvantaged_miles else 0.0
        )
        return area_share / fleet_share if fleet_share else 1.0

    def evaluate_constitutional_compliance(
        self, route_plan: RoutePlan, distances: np.ndarray, vehicle_types: List[str]
    ) -> Dict[str, Tuple[bool, float]]:
        """Evaluate the candidate against all four threshold principles."""
        burdens = list(route_plan.neighborhood_exposures.values())
        mean_burden = float(np.mean(burdens)) if burdens else 0.0
        max_burden = max(burdens, default=0.0)
        exposure_ratio = max_burden / mean_burden if mean_burden else 1.0

        served = {
            node for route in route_plan.routes for node in route
            if node != 0 and node in self.neighborhoods
        }
        service_quality = len(served) / len(self.neighborhoods) if self.neighborhoods else 1.0
        vehicle_ratio = self._vehicle_equity_ratio(
            route_plan.routes, distances, vehicle_types
        )
        if self.baseline_cost is None or self.baseline_cost <= 0.0:
            raise ValueError("A positive baseline cost is required for compliance evaluation.")
        cost_ratio = route_plan.total_cost / self.baseline_cost

        values = {
            "equitable_exposure": exposure_ratio,
            "vehicle_equity": vehicle_ratio,
            "economic_viability": cost_ratio,
            "service_quality": service_quality,
        }
        return {
            name: self.principles[name].evaluate_compliance(value)
            for name, value in values.items()
        }

    def calculate_stakeholder_utilities(self, route_plan: RoutePlan) -> Dict[str, float]:
        """Calculate dimensionless, baseline-normalized stakeholder utilities."""
        if not self.baseline_cost or not self.baseline_emissions:
            raise ValueError("Positive baseline cost and emissions are required.")
        burdens = list(route_plan.neighborhood_exposures.values())
        mean_burden = float(np.mean(burdens)) if burdens else 0.0
        max_burden = max(burdens, default=0.0)
        exposure_ratio = max_burden / mean_burden if mean_burden else 1.0
        equity_score = 1.0 / exposure_ratio if exposure_ratio else 1.0
        served = {
            node for route in route_plan.routes for node in route
            if node != 0 and node in self.neighborhoods
        }
        service_quality = len(served) / len(self.neighborhoods) if self.neighborhoods else 1.0

        utilities = {}
        for name, stakeholder in self.stakeholders.items():
            weights = stakeholder.utility_weights
            utility = 0.0
            utility -= weights.get("cost", 0.0) * route_plan.total_cost / self.baseline_cost
            utility -= weights.get("emissions", 0.0) * route_plan.total_emissions / self.baseline_emissions
            utility += weights.get("equity", 0.0) * equity_score
            utility += weights.get("service", 0.0) * service_quality
            utilities[name] = utility
        return utilities

    def _generate_candidate_plans(
        self,
        base_routes: List[List[int]],
        distances: np.ndarray,
        emissions: np.ndarray,
    ) -> List[RoutePlan]:
        """Construct baseline, equity-focused, and hybrid candidate plans."""
        equity_routes = [
            [0, 1, 5, 0], [0, 2, 6, 0], [0, 3, 7, 0], [0, 4, 8, 0]
        ]
        hybrid_routes = [
            [0, 1, 4, 0], [0, 2, 3, 0], [0, 5, 8, 0], [0, 6, 7, 0]
        ]
        candidates = [
            (1, "Cost-Weighted Baseline", base_routes, 2.0),
            (2, "Equity-Focused Candidate", equity_routes, 2.5),
            (3, "Balanced Candidate", hybrid_routes, 2.2),
        ]
        plans = []
        for plan_id, name, routes, cost_per_km in candidates:
            arc_pairs = [
                (from_node, to_node)
                for route in routes
                for from_node, to_node in zip(route[:-1], route[1:])
            ]
            total_distance = sum(float(distances[i, j]) for i, j in arc_pairs)
            total_emissions = sum(float(emissions[i, j]) for i, j in arc_pairs)
            total_cost = total_distance * cost_per_km
            burdens = self.calculate_neighborhood_exposures(routes, emissions)
            plan = RoutePlan(
                id=plan_id,
                name=name,
                routes=[route.copy() for route in routes],
                total_distance=total_distance,
                total_emissions=total_emissions,
                total_cost=total_cost,
                neighborhood_exposures=burdens,
                equity_metrics={},
                constitutional_compliance={},
                stakeholder_utilities={},
                fairness_score=0.0,
            )
            plan.fairness_score = self._calculate_fairness_score(plan)
            plans.append(plan)
        return plans

    def optimize_with_constraints(
        self,
        base_routes: List[List[int]],
        distances: np.ndarray,
        emissions: np.ndarray,
        vehicle_types: List[str],
    ) -> RoutePlan:
        """Evaluate the three prescribed candidates and apply the selection rule."""
        if len(base_routes) != len(vehicle_types):
            raise ValueError("Each route must have exactly one assigned vehicle type.")
        baseline_distance = sum(
            float(distances[i, j])
            for route in base_routes
            for i, j in zip(route[:-1], route[1:])
        )
        self.baseline_cost = baseline_distance * 2.0
        self.baseline_emissions = sum(
            float(emissions[i, j])
            for route in base_routes
            for i, j in zip(route[:-1], route[1:])
        )
        if self.baseline_cost <= 0.0 or self.baseline_emissions <= 0.0:
            raise ValueError("Baseline cost and emissions must both be positive.")

        candidates = self._generate_candidate_plans(base_routes, distances, emissions)
        evaluated = []
        for plan in candidates:
            plan.constitutional_compliance = self.evaluate_constitutional_compliance(
                plan, distances, vehicle_types
            )
            plan.stakeholder_utilities = self.calculate_stakeholder_utilities(plan)
            burdens = list(plan.neighborhood_exposures.values())
            mean_burden = float(np.mean(burdens)) if burdens else 0.0
            max_burden = max(burdens, default=0.0)
            plan.equity_metrics = {
                "mean_burden": mean_burden,
                "max_to_mean_burden_ratio": max_burden / mean_burden if mean_burden else 1.0,
            }
            penalties = sum(penalty for _, penalty in plan.constitutional_compliance.values())
            utility = sum(plan.stakeholder_utilities.values())
            plan.equity_metrics["combined_score"] = utility + 100.0 * plan.fairness_score - penalties
            is_compliant = all(
                complies for complies, _ in plan.constitutional_compliance.values()
            )
            evaluated.append((plan, penalties, is_compliant))

        compliant = [item for item in evaluated if item[2]]
        if compliant:
            selected = max(compliant, key=lambda item: item[0].equity_metrics["combined_score"])
        else:
            selected = min(
                evaluated,
                key=lambda item: (item[1], -item[0].equity_metrics["combined_score"]),
            )

        for plan, penalties, is_compliant in evaluated:
            print(f"\nPlan {plan.id}: {plan.name}")
            print(f"  Distance: {plan.total_distance:.1f} km")
            print(f"  Cost: ${plan.total_cost:.2f}")
            print(f"  Emissions: {plan.total_emissions:.1f} kg CO2")
            print(f"  Fairness score: {plan.fairness_score:.3f}")
            print(f"  Stakeholder utility sum: {sum(plan.stakeholder_utilities.values()):.3f}")
            print(f"  Constitutional penalty: {penalties:.2f} policy points")
            print(f"  Combined score: {plan.equity_metrics['combined_score']:.3f}")
            status = "compliant" if is_compliant else "not fully compliant"
            print(f"  Status: {status}")
        if not compliant:
            print("\nNo candidate satisfies every principle; selecting the least-penalized candidate.")
        return selected[0]


In [ ]:
print("CONSTITUTIONAL AI ROUTING: DATA AND STAKEHOLDER SETUP")

# Distances are kilometers; emissions are kilograms of CO2 per arc.
distances = np.array([
    [0, 8, 12, 15, 10, 18, 14, 20, 16],
    [8, 0, 6, 10, 8, 12, 10, 14, 12],
    [12, 6, 0, 8, 6, 10, 8, 12, 10],
    [15, 10, 8, 0, 5, 8, 6, 8, 10],
    [10, 8, 6, 5, 0, 6, 4, 8, 6],
    [18, 12, 10, 8, 6, 0, 5, 7, 9],
    [14, 10, 8, 6, 4, 5, 0, 6, 8],
    [20, 14, 12, 8, 8, 7, 6, 0, 10],
    [16, 12, 10, 10, 6, 9, 8, 10, 0],
], dtype=float)

emissions = np.array([
    [0, 6, 10, 12, 8, 15, 11, 18, 14],
    [6, 0, 4, 8, 6, 10, 8, 12, 10],
    [10, 4, 0, 6, 4, 8, 6, 10, 8],
    [12, 8, 6, 0, 4, 6, 5, 8, 6],
    [8, 6, 4, 4, 0, 4, 3, 6, 4],
    [15, 10, 8, 6, 4, 0, 5, 7, 9],
    [11, 8, 6, 5, 5, 0, 6, 8, 8],
    [18, 12, 10, 8, 7, 6, 6, 0, 10],
    [14, 10, 8, 6, 6, 9, 8, 10, 0],
], dtype=float)

neighborhoods = [
    Neighborhood(1, "Riverside", 5000, 85000, {"white": 0.65, "black": 0.20, "hispanic": 0.10, "asian": 0.05}, 15.0, (0.2, 0.8), 1.2),
    Neighborhood(2, "Downtown", 8000, 120000, {"white": 0.45, "black": 0.30, "hispanic": 0.15, "asian": 0.10}, 25.0, (0.5, 0.5), 1.0),
    Neighborhood(3, "Industrial Zone", 3000, 45000, {"white": 0.40, "black": 0.35, "hispanic": 0.15, "asian": 0.10}, 35.0, (0.8, 0.3), 1.5),
    Neighborhood(4, "Suburb North", 6000, 95000, {"white": 0.70, "black": 0.15, "hispanic": 0.10, "asian": 0.05}, 12.0, (0.3, 0.7), 1.0),
    Neighborhood(5, "Suburb South", 5500, 55000, {"white": 0.60, "black": 0.25, "hispanic": 0.10, "asian": 0.05}, 18.0, (0.7, 0.2), 1.3),
    Neighborhood(6, "Tech District", 4000, 110000, {"white": 0.55, "black": 0.20, "hispanic": 0.15, "asian": 0.10}, 10.0, (0.6, 0.4), 1.0),
    Neighborhood(7, "University Area", 7000, 75000, {"white": 0.60, "black": 0.20, "hispanic": 0.10, "asian": 0.10}, 8.0, (0.4, 0.6), 0.8),
    Neighborhood(8, "Historic District", 3500, 40000, {"white": 0.30, "black": 0.50, "hispanic": 0.15, "asian": 0.05}, 22.0, (0.2, 0.4), 1.4),
]

stakeholders = [
    Stakeholder("Community Residents", "residents", {"equity": 0.4, "emissions": 0.3, "cost": 0.2, "service": 0.1}, ["health", "environmental justice", "air quality"], 0.6),
    Stakeholder("Business Alliance", "businesses", {"cost": 0.5, "service": 0.3, "emissions": 0.1, "equity": 0.1}, ["profitability", "service reliability", "operational efficiency"], 0.8),
    Stakeholder("Environmental Agency", "environment", {"emissions": 0.6, "equity": 0.3, "cost": 0.05, "service": 0.05}, ["air quality", "climate change", "environmental justice"], 0.7),
    Stakeholder("City Government", "government", {"service": 0.4, "equity": 0.3, "cost": 0.2, "emissions": 0.1}, ["public health", "regulatory compliance", "political support"], 0.9),
    Stakeholder("Logistics Company", "logistics_company", {"cost": 0.6, "service": 0.25, "emissions": 0.1, "equity": 0.05}, ["profitability", "operational efficiency", "customer satisfaction"], 0.4),
]

constitutional_ai = ConstitutionalAI(neighborhoods, stakeholders)

print(f"Neighborhoods: {len(neighborhoods)}")
print(f"Stakeholders: {len(stakeholders)}")
print("Constitutional principles:")
for principle_name, principle in constitutional_ai.principles.items():
    print(f"  {principle_name}: {principle.description}")


In [ ]:
print("CONSTITUTIONAL AI CANDIDATE EVALUATION")

base_routes = [
    [0, 1, 3, 0],
    [0, 2, 4, 0],
    [0, 5, 7, 0],
    [0, 6, 8, 0],
]
vehicle_types = ["diesel", "diesel", "electric", "electric"]

assert distances.shape == emissions.shape == (9, 9)
assert len(base_routes) == len(vehicle_types)
assert all(route[0] == 0 and route[-1] == 0 for route in base_routes)
assert sorted(node for route in base_routes for node in route[1:-1]) == list(range(1, 9))

for index, route in enumerate(base_routes, start=1):
    print(f"Route {index} ({vehicle_types[index - 1]}): " + " -> ".join(map(str, route)))

optimal_plan = constitutional_ai.optimize_with_constraints(
    base_routes, distances, emissions, vehicle_types
)
final_compliance = optimal_plan.constitutional_compliance
final_utilities = optimal_plan.stakeholder_utilities
total_penalty = sum(penalty for _, penalty in final_compliance.values())
total_utility = sum(final_utilities.values())

assert set(optimal_plan.neighborhood_exposures) == set(range(1, 9))
assert np.isclose(optimal_plan.total_distance, sum(
    distances[i, j]
    for route in optimal_plan.routes
    for i, j in zip(route[:-1], route[1:])
))

print("\nSelected candidate")
print(f"Plan: {optimal_plan.name}")
print(f"Distance: {optimal_plan.total_distance:.1f} km")
print(f"Cost: ${optimal_plan.total_cost:.2f}")
print(f"Emissions: {optimal_plan.total_emissions:.1f} kg CO2")
print(f"Fairness score: {optimal_plan.fairness_score:.3f}")
print(f"Combined score: {optimal_plan.equity_metrics['combined_score']:.3f}")

print("\nConstitutional compliance")
for principle_name, (complies, penalty) in final_compliance.items():
    status = "Compliant" if complies else "Violation"
    print(f"{principle_name}: {status}; penalty = {penalty:.2f} policy points")

print("\nStakeholder utilities")
for stakeholder_name, utility in final_utilities.items():
    print(f"{stakeholder_name}: {utility:.3f}")

print("\nModeled neighborhood burden (sensitivity-weighted kg CO2 proxy)")
for n_id, burden in optimal_plan.neighborhood_exposures.items():
    neighborhood = constitutional_ai.neighborhoods[n_id]
    print(f"{neighborhood.name}: {burden:.2f}")

print(f"\nTotal stakeholder utility: {total_utility:.3f}")
print(f"Total constitutional penalty: {total_penalty:.2f} policy points")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("Algorithmic Fairness: Candidate Plan Assessment", fontsize=16, fontweight="bold")

neighborhood_names = [n.name for n in constitutional_ai.neighborhoods.values()]
burdens = [optimal_plan.neighborhood_exposures[n.id] for n in constitutional_ai.neighborhoods.values()]
x = np.arange(len(neighborhood_names))

ax = axes[0, 0]
ax.bar(x, burdens, color="firebrick", alpha=0.8, label="Modeled route-related burden")
ax.set_title("Neighborhood Burden")
ax.set_xlabel("Neighborhood")
ax.set_ylabel("Sensitivity-weighted kg CO2 proxy")
ax.set_xticks(x)
ax.set_xticklabels(neighborhood_names, rotation=45, ha="right")
ax.legend()
ax.grid(axis="y", alpha=0.3)

ax = axes[0, 1]
stakeholder_names = list(final_utilities)
utility_values = [final_utilities[name] for name in stakeholder_names]
ax.bar(stakeholder_names, utility_values, color=sns.color_palette("husl", len(stakeholder_names)))
ax.set_title("Stakeholder Utility")
ax.set_xlabel("Stakeholder")
ax.set_ylabel("Dimensionless utility")
ax.set_xticks(range(len(stakeholder_names)))
ax.set_xticklabels(stakeholder_names, rotation=35, ha="right")
ax.grid(axis="y", alpha=0.3)

ax = axes[0, 2]
principle_names = list(final_compliance)
compliance_values = [int(final_compliance[name][0]) for name in principle_names]
compliance_colors = ["seagreen" if value else "firebrick" for value in compliance_values]
ax.bar(range(len(principle_names)), compliance_values, color=compliance_colors)
ax.set_title("Constitutional Compliance")
ax.set_ylabel("Status (1 = compliant, 0 = violation)")
ax.set_ylim(0, 1.2)
ax.set_xticks(range(len(principle_names)))
ax.set_xticklabels(principle_names, rotation=35, ha="right")
ax.grid(axis="y", alpha=0.3)

candidate_plans = constitutional_ai._generate_candidate_plans(base_routes, distances, emissions)
for plan in candidate_plans:
    plan.fairness_score = constitutional_ai._calculate_fairness_score(plan)

ax = axes[1, 0]
candidate_costs = [plan.total_cost for plan in candidate_plans]
candidate_fairness = [plan.fairness_score for plan in candidate_plans]
ax.scatter(candidate_costs, candidate_fairness, s=90, color="darkslateblue")
for plan, cost, score in zip(candidate_plans, candidate_costs, candidate_fairness):
    ax.annotate(plan.name, (cost, score), xytext=(5, 5), textcoords="offset points", fontsize=8)
ax.set_title("Candidate Cost and Fairness")
ax.set_xlabel("Total cost ($)")
ax.set_ylabel("Fairness score (0–1; higher is more even)")
ax.grid(alpha=0.3)

ax = axes[1, 1]
ax.bar(stakeholder_names, utility_values, color=sns.color_palette("husl", len(stakeholder_names)))
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Stakeholder Utility Distribution")
ax.set_xlabel("Stakeholder")
ax.set_ylabel("Dimensionless utility")
ax.set_xticks(range(len(stakeholder_names)))
ax.set_xticklabels(stakeholder_names, rotation=35, ha="right")
ax.grid(axis="y", alpha=0.3)

ax = axes[1, 2]
coords = np.array([n.geographic_coords for n in constitutional_ai.neighborhoods.values()])
scatter = ax.scatter(coords[:, 0], coords[:, 1], c=burdens, cmap="Reds", s=220, edgecolors="black")
for neighborhood, (x_coord, y_coord), burden in zip(
    constitutional_ai.neighborhoods.values(), coords, burdens
):
    ax.annotate(f"{neighborhood.id}: {burden:.1f}", (x_coord, y_coord), xytext=(5, 5), textcoords="offset points", fontsize=8)
ax.set_title("Spatial Burden Illustration")
ax.set_xlabel("X coordinate (illustrative)")
ax.set_ylabel("Y coordinate (illustrative)")
colorbar = fig.colorbar(scatter, ax=ax)
colorbar.set_label("Sensitivity-weighted kg CO2 proxy")

fig.tight_layout()
plt.show()


In [ ]:
print("ALGORITHMIC FAIRNESS ANALYSIS")

compliant_count = sum(complies for complies, _ in final_compliance.values())
high_burden_threshold = 1.2 * optimal_plan.equity_metrics["mean_burden"]
high_burden_neighborhoods = [
    constitutional_ai.neighborhoods[n_id].name
    for n_id, burden in optimal_plan.neighborhood_exposures.items()
    if burden > high_burden_threshold
]

print("\n1. Principle assessment")
print(f"  Principles met: {compliant_count}/{len(final_compliance)}")
for name, (complies, penalty) in final_compliance.items():
    status = "compliant" if complies else "violation"
    print(f"  {name}: {status}; penalty {penalty:.2f} policy points")

print("\n2. Modeled burden distribution")
print(f"  Fairness score: {optimal_plan.fairness_score:.3f} (higher indicates a more even distribution)")
print(f"  Neighborhoods above 120% of mean modeled burden: {high_burden_neighborhoods}")
print("  This emissions-based proxy is not a measurement of local PM2.5 concentration.")

print("\n3. Stakeholder utility")
print(f"  Sum of stakeholder utilities: {total_utility:.3f}")
print(f"  Highest utility: {max(final_utilities, key=final_utilities.get)}")
print(f"  Lowest utility: {min(final_utilities, key=final_utilities.get)}")

print("\n4. Cost and emissions relative to the baseline routes")
cost_change = 100.0 * (optimal_plan.total_cost / constitutional_ai.baseline_cost - 1.0)
emissions_change = 100.0 * (optimal_plan.total_emissions / constitutional_ai.baseline_emissions - 1.0)
print(f"  Cost change: {cost_change:+.1f}%")
print(f"  Emissions change: {emissions_change:+.1f}%")
print("  These comparisons use the stated baseline routes and the baseline cost rate.")

print("\n5. Interpretation and limitations")
print("  The selected plan is the best candidate under the stated selection rule, not a proven global optimum.")
print("  Candidate routes, fixed arc emissions, threshold choices, and stakeholder weights shape the result.")
print("  The model does not establish regulatory compliance or predict community acceptance.")
print("  A practical assessment requires validated local exposure, fleet, service, and stakeholder data.")


## Key interpretations

- **Threshold rules and selection are separate steps.** Each candidate is first evaluated against every principle. The selection rule then favors the highest combined score among compliant candidates; if there are none, it minimizes total penalty before using the score as a tie-breaker.
- **Fairness is operationalized, not asserted.** The Gini-based score summarizes inequality in the model's sensitivity-weighted allocation of supplied route emissions. It does not measure real-world air quality or capture every dimension of environmental justice.
- **Stakeholder utilities depend on modeling choices.** Their terms are normalized against the baseline cost and emissions so their scales are more comparable. The chosen weights still encode value judgments, and the reported utility sum is not a universal measure of social welfare.
- **Vehicle equity is distinct from emissions accounting here.** Electric miles are used to assess their allocation to disadvantaged areas, while the arc-emissions matrix remains fixed regardless of vehicle type. Consequently, this example does not estimate the emissions reduction from electrification.
- **The candidate set bounds what can be concluded.** The procedure compares only the three constructed plans. Its selected candidate is not guaranteed to be the best route plan outside that set.

## Summary

This example combines explicit threshold principles, stakeholder-weighted utilities, and a transparent candidate-selection rule. Its educational value lies in making the assumptions, metrics, penalties, and trade-offs inspectable. Applying the framework to operational planning would require validated emissions and exposure data, justified policy thresholds, meaningful community participation, and a route-generation method capable of representing the relevant operational constraints.